In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/VuTrinhNguyenHoang/bidirectional-table-text-alignment.git

In [ ]:
%cd bidirectional-table-text-alignment

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
MODE = "small"  # "small", "medium", "debug", "full"
GENERATOR_MODE = "full" 
SELECTOR_MODE = "small"
CONFIG = "configs/main.yaml"

In [ ]:
!PYTHONPATH=. python scripts/prepare_subset.py --mode {MODE} --generator-mode {GENERATOR_MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/tokenize_dataset.py --mode {MODE} --generator-mode {GENERATOR_MODE} --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/train_generator.py --mode {MODE} --generator-mode {GENERATOR_MODE} --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/evaluation_generation.py --mode {MODE} --generator-mode {GENERATOR_MODE} --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/run_faithfulness.py --mode {MODE} --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/prepare_cell_dataset.py --mode {MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/tokenize_cell_dataset.py --mode {MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/train_cell_selector.py --mode {MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG}

In [ ]:
!PYTHONPATH=. python scripts/evaluation_cell_selector.py --mode {MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k 3

In [ ]:
!PYTHONPATH=. python scripts/evaluation_cell_selector.py --mode {MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k 5

In [ ]:
!PYTHONPATH=. python scripts/evaluation_cell_selector.py --mode {MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k 7

In [ ]:
!PYTHONPATH=. python scripts/evaluation_e2e.py --mode {MODE} --generator-mode {GENERATOR_MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k 3

In [ ]:
!PYTHONPATH=. python scripts/evaluation_e2e.py --mode {MODE} --generator-mode {GENERATOR_MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k 5

In [ ]:
!PYTHONPATH=. python scripts/evaluation_e2e.py --mode {MODE} --generator-mode {GENERATOR_MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k 7

In [ ]:
!ls -R outputs/metrics/{MODE}
!ls -R outputs/predictions/{MODE}
!ls -R outputs/checkpoints

In [ ]:
import json
from pathlib import Path

metric_dir = Path(f"outputs/metrics/{MODE}")

for path in sorted(metric_dir.glob("*.json")):
    print("=" * 80)
    print(path)
    with open(path, "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

In [ ]:
import json
from pathlib import Path

path = Path(f"outputs/predictions/{MODE}/e2e_predictions_top3.jsonl")

with open(path, "r", encoding="utf-8") as f:
    for i, line in zip(range(5), f):
        record = json.loads(line)
        print("=" * 100)
        print("TARGET:", record["target"])
        print("PRED  :", record["prediction"])
        print("GOLD CELLS:", record["gold_highlighted_cells"])
        print("PRED CELLS:", record["pred_highlighted_cells"])
        print("CELL METRICS:", record["cell_metrics"])
        print("FAITHFULNESS:", record["faithfulness"])

In [ ]:
!PYTHONPATH=. python scripts/make_report_plots.py --mode {MODE} --generator-mode {GENERATOR_MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG}

In [ ]:
!find outputs/plots/{MODE} -type f | sort

In [ ]:
from IPython.display import Image, display
from pathlib import Path

plot_dir = Path(f"outputs/plots/{MODE}")

for path in sorted(plot_dir.glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))